# Movie Recommendation System — Corrected MLP

This reproducibility-focused revision retains the original **TensorFlow/Keras user/movie embedding + MLP** approach. It trains on MovieLens ratings only because `ratings_small.movieId` must **not** be directly joined to `movies_metadata.id` (TMDb ID). Metadata titles are optional and joined only via `links_small.csv`.

**Evaluation:** train/validation/test split first; training-only ID mappings; held-out warm-start metrics; baseline comparison. Historical MAE/MSE from the original notebook are not claimed as reproduced.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
DATA = Path("data")
ratings_path = DATA / "ratings_small.csv"
if not ratings_path.exists():
    raise FileNotFoundError("Download ratings_small.csv from Kaggle The Movies Dataset and save it to data/")
ratings = pd.read_csv(ratings_path, usecols=["userId", "movieId", "rating"])
ratings = ratings.dropna().drop_duplicates(["userId", "movieId"]).copy()
ratings[["userId", "movieId"]] = ratings[["userId", "movieId"]].astype("int64")
ratings["rating"] = ratings["rating"].astype("float32")
print("Ratings:", len(ratings), "| users:", ratings.userId.nunique(), "| movies:", ratings.movieId.nunique())
display(ratings.head())

## Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ratings["rating"].hist(bins=np.arange(0.25, 5.76, 0.5), ax=ax)
ax.set(xlabel="Rating", ylabel="Number of ratings", title="MovieLens rating distribution")
plt.show()
print(ratings["rating"].describe())

## Split before fitting encoders

Hold out 10% for testing and 10% for validation. Encoders are derived **only from training data**. Unknown users/movies map to index 0, which is reserved for unseen IDs. We report warm-start metrics separately because an embedding model cannot learn individual representations for IDs absent from training.

In [ ]:
train_df, holdout_df = train_test_split(ratings, test_size=0.20, random_state=SEED)
val_df, test_df = train_test_split(holdout_df, test_size=0.50, random_state=SEED)
user_to_idx = {int(u): i + 1 for i, u in enumerate(sorted(train_df.userId.unique()))}
movie_to_idx = {int(m): i + 1 for i, m in enumerate(sorted(train_df.movieId.unique()))}
def encode(frame):
    return [
        frame.userId.map(user_to_idx).fillna(0).to_numpy(dtype="int32"),
        frame.movieId.map(movie_to_idx).fillna(0).to_numpy(dtype="int32")
    ]
def warm_mask(frame):
    return frame.userId.isin(user_to_idx) & frame.movieId.isin(movie_to_idx)
train_x, val_x, test_x = encode(train_df), encode(val_df), encode(test_df)
train_y = train_df.rating.to_numpy(dtype="float32")
val_y = val_df.rating.to_numpy(dtype="float32")
test_y = test_df.rating.to_numpy(dtype="float32")
print("Split sizes:", len(train_df), len(val_df), len(test_df))
print("Warm-start test rows:", int(warm_mask(test_df).sum()), "/", len(test_df))

## User/movie embedding MLP

Only user and movie IDs are used as model features. This avoids introducing incorrectly matched TMDb metadata. Ratings are clipped to the valid 0.5–5 scale at evaluation time.

In [ ]:
user_input = layers.Input(shape=(1,), dtype="int32", name="user")
movie_input = layers.Input(shape=(1,), dtype="int32", name="movie")
user_vec = layers.Flatten()(layers.Embedding(len(user_to_idx) + 1, 32)(user_input))
movie_vec = layers.Flatten()(layers.Embedding(len(movie_to_idx) + 1, 32)(movie_input))
x = layers.Concatenate()([user_vec, movie_vec])
x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dense(64, activation="relu")(x)
out = layers.Dense(1)(x)
model = Model([user_input, movie_input], out)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"])
history = model.fit(
    train_x, train_y,
    validation_data=(val_x, val_y),
    epochs=20, batch_size=256, verbose=1,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
)

## Held-out evaluation and baseline

Metrics are calculated only after training. A training-set mean-rating predictor provides a simple comparison. The main model metric uses test rows where both IDs were present during training; cold-start coverage is reported explicitly.

In [ ]:
mask = warm_mask(test_df).to_numpy()
if not mask.any():
    raise RuntimeError("No warm-start test examples available.")
test_x_warm = [x[mask] for x in test_x]
actual = test_y[mask]
pred = np.clip(model.predict(test_x_warm, verbose=0).reshape(-1), 0.5, 5.0)
baseline = np.full_like(actual, train_y.mean())
results = pd.DataFrame([
    {"Model": "Training-mean baseline", "MAE": mean_absolute_error(actual, baseline), "MSE": mean_squared_error(actual, baseline)},
    {"Model": "Embedding MLP", "MAE": mean_absolute_error(actual, pred), "MSE": mean_squared_error(actual, pred)}
])
display(results.round(4))
print(f"Warm-start coverage: {mask.mean():.1%}; unseen-ID rows: {(~mask).sum()}")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history["loss"], label="Training MSE")
ax.plot(history.history["val_loss"], label="Validation MSE")
ax.set(xlabel="Epoch", ylabel="MSE loss", title="Learning curves")
ax.legend()
plt.show()

## Optional movie titles with correct ID mapping

To display titles, download **both** `links_small.csv` and `movies_metadata.csv` from the same Kaggle dataset. `links_small.movieId` is the MovieLens ID; `links_small.tmdbId` maps to `movies_metadata.id`. Without these files, recommendations display MovieLens IDs instead of inventing titles.

In [ ]:
movie_titles = {}
links_path = DATA / "links_small.csv"
metadata_path = DATA / "movies_metadata.csv"
if links_path.exists() and metadata_path.exists():
    links = pd.read_csv(links_path, usecols=["movieId", "tmdbId"])
    meta = pd.read_csv(metadata_path, usecols=["id", "title"], low_memory=False)
    links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce")
    meta["id"] = pd.to_numeric(meta["id"], errors="coerce")
    title_lookup = (links.dropna(subset=["tmdbId"])
                    .merge(meta.dropna(subset=["id"]), left_on="tmdbId", right_on="id", how="left")
                    .dropna(subset=["title"]).drop_duplicates("movieId"))
    movie_titles = dict(zip(title_lookup.movieId.astype(int), title_lookup.title))
    print(f"Matched titles: {len(movie_titles)}")
else:
    print("Optional title files not found. MovieLens IDs will be shown.")

## Recommendations

For an existing training user, score **all movies seen during training** and exclude movies the user has already rated anywhere in the provided ratings data. New users receive popular movies ranked by count and mean rating from the training set; this is a simple cold-start fallback.

In [ ]:
def recommend_movies(user_id, top_n=5):
    user_id = int(user_id)
    if top_n < 1:
        raise ValueError("top_n must be positive")
    rated = set(ratings.loc[ratings.userId.eq(user_id), "movieId"].astype(int))
    candidates = np.array([m for m in movie_to_idx if m not in rated], dtype="int64")
    if user_id in user_to_idx and len(candidates):
        user_vec = np.full(len(candidates), user_to_idx[user_id], dtype="int32")
        movie_vec = np.array([movie_to_idx[int(m)] for m in candidates], dtype="int32")
        scores = np.clip(model.predict([user_vec, movie_vec], verbose=0).reshape(-1), 0.5, 5.0)
        output = pd.DataFrame({"movieId": candidates, "score": scores})
        output = output.sort_values("score", ascending=False).head(top_n)
    else:
        popularity = train_df.groupby("movieId").rating.agg(["mean", "count"])
        popularity = popularity.loc[~popularity.index.isin(rated)]
        # Minimum evidence threshold; fall back to all candidates for small datasets.
        eligible = popularity[popularity["count"] >= 5]
        if eligible.empty:
            eligible = popularity
        output = (eligible.sort_values(["count", "mean"], ascending=False)
                  .head(top_n).reset_index()[["movieId", "mean"]]
                  .rename(columns={"mean": "score"}))
    output["title"] = output.movieId.map(movie_titles).fillna("Title unavailable")
    return output[["movieId", "title", "score"]].reset_index(drop=True)

example_user = int(train_df.userId.iloc[0])
display(recommend_movies(example_user, top_n=5))
display(recommend_movies(-1, top_n=5))

## Limitations and next steps

- This is a rating-prediction recommender, not a production-grade ranking system.
- A random interaction split measures warm-start rating prediction; it does not simulate future recommendations. A chronological split is a useful extension.
- Unknown IDs have no learned embeddings; the reported model evaluation excludes such test rows.
- Title enrichment is optional and requires a verified MovieLens-to-TMDb link.
- Results are intentionally not hard-coded. Run all cells and report the metrics obtained in your environment.